# Chapter 4: Multiple Regression Analysis — Inference

Topics: t-test, confidence intervals, F-tests for linear restrictions.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
import scipy.stats as stats
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)

## 4.1 The t-Test

In [ ]:
df = woo.dataWoo('wage1')
model = smf.ols('wage ~ educ + exper + tenure + female', data=df).fit()

print("Coefficients, t-stats, and p-values:")
summary_df = pd.DataFrame({
    'coef': model.params,
    't-stat': model.tvalues,
    'p-value': model.pvalues,
    'sig': ['***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.1 else ''
            for p in model.pvalues]
})
print(summary_df.round(4))

## 4.2 Confidence Intervals

In [ ]:
ci = model.conf_int(alpha=0.05)
ci.columns = ['2.5%', '97.5%']
ci['coef'] = model.params
print("95% Confidence Intervals:")
print(ci[['coef','2.5%','97.5%']].round(4))

# Plot CIs
fig, ax = plt.subplots(figsize=(8, 5))
params = model.params.drop('Intercept')
lower  = ci['2.5%'].drop('Intercept')
upper  = ci['97.5%'].drop('Intercept')
ax.barh(params.index, params, xerr=[params - lower, upper - params],
        color='steelblue', alpha=0.7, capsize=5)
ax.axvline(0, color='red', linestyle='--')
ax.set_title('Coefficient Estimates with 95% CIs')
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_confidence_intervals.png", dpi=150)
plt.show()

## 4.3 Linear Restrictions: F-Tests

In [ ]:
# Test: H0: β_exper = 0 AND β_tenure = 0
f_test = model.f_test("exper = 0, tenure = 0")
print(f"F-statistic: {f_test.fvalue[0][0]:.4f}")
print(f"p-value:     {f_test.pvalue:.4f}")

# Overall F-test (all slopes = 0)
print(f"\nOverall F-stat: {model.fvalue:.4f}")
print(f"Overall F p-value: {model.f_pvalue:.4f}")